# Exploratory Data Analysis - Chocolate Ratings Dataset

In [ ]:
!pip install pycountry pycountry-convert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 12.8 MB/s eta 0:00:00


In [ ]:
# All Imports
import pandas as pd
import numpy as np
import ast
import pycountry_convert as pc

In [ ]:
df = pd.read_csv('chocolate_ratings.csv')
df.head()

,REF,Company (Manufacturer),Company Location,Review Date,Country of Bean Origin,Specific Bean Origin or Bar Name,Cocoa Percent,Ingredients,Most Memorable Characteristics,Rating
0,2454,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76%,"3- B,S,C","rich cocoa, fatty, bready",3.25
1,2458,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76%,"3- B,S,C","cocoa, vegetal, savory",3.50
2,2454,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76%,"3- B,S,C","cocoa, blackberry, full body",3.75
3,2542,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68%,"3- B,S,C","chewy, off, rubbery",3.00
4,2546,5150,U.S.A.,2021,Venezuela,"Sur del Lago, batch 1",72%,"3- B,S,C","fatty, earthy, moss, nutty,chalky",3.00


In [ ]:
df.shape

(2530, 10)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2530 entries, 0 to 2529
Data columns (total 10 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   REF                               2530 non-null   int64  
 1   Company (Manufacturer)            2530 non-null   object 
 2   Company Location                  2530 non-null   object 
 3   Review Date                       2530 non-null   int64  
 4   Country of Bean Origin            2530 non-null   object 
 5   Specific Bean Origin or Bar Name  2530 non-null   object 
 6   Cocoa Percent                     2530 non-null   object 
 7   Ingredients                       2443 non-null   object 
 8   Most Memorable Characteristics    2530 non-null   object 
 9   Rating                            2530 non-null   float64
dtypes: float64(1), int64(2), object(7)
memory usage: 197.8+ KB


Ingredients are missing therefore require handling. Since the missing values are in a minority it is just best to remove them rather than assuming values and altering potential findings

In [ ]:
df = df.dropna(subset=['Ingredients'])

In [ ]:
df = df.reset_index(drop=True)

Splitting the Ingredients column on the '-' & picking the first element to make a new column containing the number of ingredients

In [ ]:
df['Ingredient Count'] = df['Ingredients'].apply(lambda x: int(x.split('-')[0]))

Splitting the Ingredients column once again on the '-' & picking the second element to make a new column containing the ingredient initials

In [ ]:
df['Ingredient_Initials'] = df['Ingredients'].str.split('-').str[1].str.strip()

In [ ]:
df.head()

,REF,Company (Manufacturer),Company Location,Review Date,Country of Bean Origin,Specific Bean Origin or Bar Name,Cocoa Percent,Ingredients,Most Memorable Characteristics,Rating,Ingredient Count,Ingredient_Initials
0,2454,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76%,"3- B,S,C","rich cocoa, fatty, bready",3.25,3,"B,S,C"
1,2458,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76%,"3- B,S,C","cocoa, vegetal, savory",3.50,3,"B,S,C"
2,2454,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76%,"3- B,S,C","cocoa, blackberry, full body",3.75,3,"B,S,C"
3,2542,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68%,"3- B,S,C","chewy, off, rubbery",3.00,3,"B,S,C"
4,2546,5150,U.S.A.,2021,Venezuela,"Sur del Lago, batch 1",72%,"3- B,S,C","fatty, earthy, moss, nutty,chalky",3.00,3,"B,S,C"


Subsectioning cocoa percentages into brackets

In [ ]:
df['Cocoa Percent'] = df['Cocoa Percent'].str.replace('%', '').astype(float)

In [ ]:
df['Cocoa Percent'].isna().sum()

np.int64(0)

In [ ]:
df = df[df['Cocoa Percent'] >= 50].reset_index(drop=True)

In [ ]:
bin_edges = np.linspace(50, 100, 6)
labels = ['50-60%', '60-70%', '70-80%', '80-90%', '90-100%']

df['Cocoa Bin'] = pd.cut(df['Cocoa Percent'], bins=bin_edges, labels=labels, include_lowest=True)

In [ ]:
# Drop rows where Country of Bean Origin is missing
df = df.dropna(subset=['Country of Bean Origin']).reset_index(drop=True)
df['Country of Bean Origin'] = df['Country of Bean Origin'].str.strip().str.title()

In [ ]:
def clean_country(country):
    if pd.isna(country):
        return "Unknown"

    country = country.strip()

    # Handle multiple countries
    if ',' in country:
        country = country.split(',')[0]

    return country

df['CleanCountry'] = df['Country of Bean Origin'].apply(clean_country)

# Mapping Continents
special_country_map = {
    "burma": "Asia",
    "sao tome": "Africa",
    "sao tome & principe": "Africa",
    "trinidad": "North America",
    "blend": "Other",
    "u.s.a.": "North America",
    "st.vincent-grenadines": "North America",
    "dr congo": "Africa"
}

def get_continent(country):
    country_lower = country.lower().strip()

    # Handling special cases
    if country_lower in special_country_map:
        return special_country_map[country_lower]

    try:
        country_alpha2 = pc.country_name_to_country_alpha2(country)
        continent_code = pc.country_alpha2_to_continent_code(country_alpha2)

        continent_map = {
            'AF': 'Africa',
            'AS': 'Asia',
            'EU': 'Europe',
            'NA': 'North America',
            'SA': 'South America',
            'OC': 'Oceania'
        }

        return continent_map.get(continent_code, "Other")

    except:
        return "Other"


df['Continent'] = df['CleanCountry'].apply(get_continent)

print(df[['Country of Bean Origin', 'CleanCountry', 'Continent']].head(20))
print("\nContinent distribution:\n", df['Continent'].value_counts())

   Country of Bean Origin        CleanCountry      Continent
0                Tanzania            Tanzania         Africa
1      Dominican Republic  Dominican Republic  North America
2              Madagascar          Madagascar         Africa
3                    Fiji                Fiji        Oceania
4               Venezuela           Venezuela  South America
5                  Uganda              Uganda         Africa
6                   India               India           Asia
7                 Bolivia             Bolivia  South America
8                    Peru                Peru  South America
9                  Panama              Panama  North America
10               Colombia            Colombia  South America
11             Madagascar          Madagascar         Africa
12                  Burma               Burma           Asia
13                 Brazil              Brazil  South America
14       Papua New Guinea    Papua New Guinea        Oceania
15                   Per

Keeping only relevant columns

In [ ]:
columns_map = {
    'Company (Manufacturer)': 'Company',
    'Company Location': 'CompanyLocation',
    'Review Date': 'Year',
    'Country of Bean Origin': 'BeanOrigin',
    'Continent':'Continent',
    'Cocoa Percent': 'CocoaPercent',
    'Cocoa Bin': 'CocoaBin',
    'Ingredient Count': 'IngredientCount',
    'Ingredient_Initials': 'IngredientInitials',
    'Rating': 'Rating'
}

df_clean = df[list(columns_map.keys())].rename(columns=columns_map)

In [ ]:
df_clean.to_csv("cleaned_ratings.csv", index=False)

Processing an additional column to produce most memorable characteristics. This processing is stored in another file as it makes use of exploding data and would not support any other visualisation type apart from a word cloud

In [ ]:
df = df.dropna(subset=['Most Memorable Characteristics'])

df['Most Memorable Characteristics'] = df['Most Memorable Characteristics'].str.lower()

df['CharacteristicList'] = df['Most Memorable Characteristics'].str.split(',')

df['CharacteristicList'] = df['CharacteristicList'].apply(
    lambda x: [i.strip() for i in x]
)

df_exploded = df.explode('CharacteristicList')

df_exploded = df_exploded.rename(columns={
    'CharacteristicList': 'Characteristic'
})

df_exploded = df_exploded[df_exploded['Characteristic'] != '']

counts = df_exploded['Characteristic'].value_counts()

# Keeping only characteristics appearing at least 5 times
valid_chars = counts[counts >= 5].index

df_exploded = df_exploded[df_exploded['Characteristic'].isin(valid_chars)]

In [ ]:
print(df_exploded['Characteristic'].value_counts().head(10))

Characteristic
sweet      259
nutty      256
cocoa      242
roasty     213
creamy     187
earthy     181
sandy      164
fatty      161
floral     141
intense    139
Name: count, dtype: int64


Removing unnecesary columns

In [ ]:
columns_map = {
    'Company (Manufacturer)': 'Company',
    'Company Location': 'CompanyLocation',
    'Review Date': 'Year',
    'Country of Bean Origin': 'BeanOrigin',
    'Continent':'Continent',
    'Cocoa Percent': 'CocoaPercent',
    'Cocoa Bin': 'CocoaBin',
    'Ingredient Count': 'IngredientCount',
    'Ingredient_Initials': 'IngredientInitials',
    'Characteristic': 'Characteristic',
    'Rating': 'Rating'
}

df_clean = df_exploded[list(columns_map.keys())].rename(columns=columns_map)

In [ ]:
df_clean.head()

,Company,CompanyLocation,Year,BeanOrigin,Continent,CocoaPercent,CocoaBin,IngredientCount,IngredientInitials,Characteristic,Rating
0,5150,U.S.A.,2019,Tanzania,Africa,76.0,70-80%,3,"B,S,C",rich cocoa,3.25
0,5150,U.S.A.,2019,Tanzania,Africa,76.0,70-80%,3,"B,S,C",fatty,3.25
0,5150,U.S.A.,2019,Tanzania,Africa,76.0,70-80%,3,"B,S,C",bready,3.25
1,5150,U.S.A.,2019,Dominican Republic,North America,76.0,70-80%,3,"B,S,C",cocoa,3.50
1,5150,U.S.A.,2019,Dominican Republic,North America,76.0,70-80%,3,"B,S,C",vegetal,3.50


In [ ]:
df_clean.to_csv("exploded_ratings.csv", index=False)